# Repeat the comparison between b-shells average signal across sessions, but, sample only 32 directions * shell in order to keep the number of directions constant: 
------------------------------------------------
I made 3 samples for b=1000 and 2 samples for other shells 

In [ ]:
import re
import numpy as np
import os
import nibabel as nib
from nilearn import image as nlimage
import tempfile

vps = [i for i in range(44) if i not in [32]]
sessions = ["ses-01", "ses-02"]
APs = ["A", "B", "C"]
SMOOTH_FWHM_MM = "6"
TEMPLATE = "MNI"
N_PER_OCTANT = 8
RNG_SEED = 42

pid = "sub-00"
ses = "ses-01"
subjid = f"{pid}_{ses}"

home = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"
rng = np.random.default_rng(RNG_SEED)

AP = "A"
dvs_path = f"/home/malberti/wks14/temp/FF_DWI_Drift/DWI_plit_{AP}.dvs"
bvals = os.path.join(home, pid, ses, "dwi", "signal_drift", f"{subjid}_dwi_eddy_corrected_{AP}_noPA.bval")

BVALs = np.loadtxt(bvals)

vectors = []
with open(dvs_path) as f:
    for line in f:
        match = re.match(r"Vector\[\d+\]\s*=\s*\(([-\d.]+),([-\d.]+),([-\d.]+)\)", line)
        if match:
            vectors.append([float(match.group(1)), float(match.group(2)), float(match.group(3))])

vectors = np.array(vectors)
print(f"[INFO] Parsed {len(vectors)} | {len(BVALs)} vectors from {dvs_path}")

if len(vectors) == len(BVALs) - 1:
    vectors = np.vstack([[0.0, 0.0, 0.0], vectors])
    print(f"[INFO] Padded vectors: {len(vectors)} == {len(BVALs)}")
elif len(vectors) != len(BVALs):
    raise ValueError(f"Unexpected mismatch: vectors={len(vectors)}, BVALs={len(BVALs)}")

indices = np.arange(len(BVALs))
VECTORs = np.column_stack([vectors, BVALs, indices])
tolerance = 50

SHELLs = np.unique(np.round(BVALs / tolerance) * tolerance)

# build resamples per shell: b<550 -> 1 sample, b==1000 -> 3 samples, b>1200 -> 2 samples
Partitioned_SHELLs = {}  # {shell: [octant_dict, octant_dict, ...]}

for shell in SHELLs:
    used_indices = set()

    if shell < 550:
        n_samples = 1
    elif shell == 1000:
        n_samples = 3
    elif shell > 1200:
        n_samples = 2
    else:
        n_samples = 1

    samples = []

    for i in range(n_samples):
        bSHELL_Vectors = VECTORs[(VECTORs[:, 3] > shell - tolerance) & (VECTORs[:, 3] < shell + tolerance)]
        bSHELL_Vectors = bSHELL_Vectors[~np.isin(bSHELL_Vectors[:, 4], list(used_indices))]

        FIRST_Oct = bSHELL_Vectors[
            ((bSHELL_Vectors[:, 0] > 0) & (bSHELL_Vectors[:, 1] < 0) & (bSHELL_Vectors[:, 2] > 0)) |
            ((bSHELL_Vectors[:, 0] < 0) & (bSHELL_Vectors[:, 1] > 0) & (bSHELL_Vectors[:, 2] < 0))
        ]
        SECOND_Oct = bSHELL_Vectors[
            ((bSHELL_Vectors[:, 0] > 0) & (bSHELL_Vectors[:, 1] < 0) & (bSHELL_Vectors[:, 2] < 0)) |
            ((bSHELL_Vectors[:, 0] < 0) & (bSHELL_Vectors[:, 1] > 0) & (bSHELL_Vectors[:, 2] > 0))
        ]
        THIRD_Oct = bSHELL_Vectors[
            ((bSHELL_Vectors[:, 0] > 0) & (bSHELL_Vectors[:, 1] > 0) & (bSHELL_Vectors[:, 2] > 0)) |
            ((bSHELL_Vectors[:, 0] < 0) & (bSHELL_Vectors[:, 1] < 0) & (bSHELL_Vectors[:, 2] < 0))
        ]
        FOURTH_Oct = bSHELL_Vectors[
            ((bSHELL_Vectors[:, 0] > 0) & (bSHELL_Vectors[:, 1] > 0) & (bSHELL_Vectors[:, 2] < 0)) |
            ((bSHELL_Vectors[:, 0] < 0) & (bSHELL_Vectors[:, 1] < 0) & (bSHELL_Vectors[:, 2] > 0))
        ]

        octants = {}
        for label, oct_data in [("FIRST_Oct", FIRST_Oct), ("SECOND_Oct", SECOND_Oct), ("THIRD_Oct", THIRD_Oct), ("FOURTH_Oct", FOURTH_Oct)]:
            oct_indices = oct_data[:, 4].astype(int)
            if len(oct_indices) > N_PER_OCTANT:
                oct_indices = rng.choice(oct_indices, size=N_PER_OCTANT, replace=False)
            elif len(oct_indices) < N_PER_OCTANT:
                print(f"[WARN] shell {int(shell)} {label}: only {len(oct_indices)} directions available, wanted {N_PER_OCTANT}")
            octants[label] = oct_indices

        used_indices.update(np.concatenate(list(octants.values())).tolist())
        samples.append(octants)

    Partitioned_SHELLs[shell] = samples
    print(f"[INFO] shell {int(shell)}: {n_samples} resample(s) built")

# recap: exact volume/index/bval used per shell/resample/octant
recap_lines = ["# Volume / Index / Bval correspondence\n", f"AP direction: {AP}\n"]
for shell, samples in Partitioned_SHELLs.items():
    for sample_i, octants in enumerate(samples):
        recap_lines.append(f"\n## Shell b{int(shell)} - resample {sample_i}\n")
        for label, vol_indices in octants.items():
            recap_lines.append(f"\n### {label}\n")
            recap_lines.append("| volume_index | bval | x | y | z |\n")
            recap_lines.append("|---|---|---|---|---|\n")
            for idx in vol_indices:
                x, y, z = vectors[idx]
                recap_lines.append(f"| {idx} | {BVALs[idx]:.1f} | {x:.4f} | {y:.4f} | {z:.4f} |\n")
recap_content = "".join(recap_lines)

out_home = r"/home/malberti/Unix_Folders/SWEEP2/Gradients_Stability_Analysis"
out_dir = os.path.join(out_home, "Gradients_Stability_Quadrants")
os.makedirs(out_dir, exist_ok=True)

for vp in vps:
    for session in sessions:
        pid = f"sub-{vp:02d}"
        subjid = f"{pid}_{session}"

        dwi_path = os.path.join(out_home, "Volume_MNI", f"{subjid}_{AP}_desc-MNI.nii.gz")
        dwi_img = nib.load(dwi_path)

        with tempfile.TemporaryDirectory() as tmp:
            smoothed = nlimage.smooth_img(dwi_img, fwhm=float(SMOOTH_FWHM_MM))
            dwi_out = os.path.join(tmp, f"{subjid}_{AP}_desc-MNI{SMOOTH_FWHM_MM}mm.nii.gz")
            smoothed.to_filename(dwi_out)
            dwi_img = nib.load(dwi_out)

            dwi_affine = dwi_img.affine
            dwi_header = dwi_img.header
            dwi_data = dwi_img.get_fdata()

            recap_path = os.path.join(out_dir, f"{subjid}_{AP}_recap_readme.md")
            with open(recap_path, "w") as f:
                f.write(recap_content)

            per_shell_avgs = []  # one volume per shell, each = avg of that shell's octants/resamples

            for shell, samples in Partitioned_SHELLs.items():
                shell_components = []
                for octants in samples:
                    for label, vol_indices in octants.items():
                        quadrant_avg = np.mean(dwi_data[..., vol_indices], axis=3)
                        shell_components.append(quadrant_avg)
                shell_avg = np.mean(np.stack(shell_components, axis=-1), axis=-1)
                per_shell_avgs.append(shell_avg)

            final_avg = np.mean(np.stack(per_shell_avgs, axis=-1), axis=-1)  # equal weight per shell

            out_path = os.path.join(out_dir, f"{subjid}_{AP}_desc-MNI{SMOOTH_FWHM_MM}mm_avg.nii.gz")
            nib.Nifti1Image(final_avg, dwi_affine, dwi_header).to_filename(out_path)
            print(f"[INFO] Saved {out_path}  shape={final_avg.shape}")

print("[INFO] Done")